In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
%run ../../config/utils

In [0]:
import sys
sys.path.append("..")
sys.path.append('../..')
from datetime import datetime

import os
import yaml
from pyspark.sql import functions as f
from urllib.parse import urlparse
import lib_trip_spend.spark_general_utilities as util_func

In [0]:
today_str = datetime.today().strftime("%Y-%m-%d")
dbutils.widgets.text("run_as_date", today_str, "Date for data processing") # create the widget if missing

In [0]:
date_of_run_str = dbutils.widgets.get("run_as_date")
run_as_date = datetime.strptime(date_of_run_str, "%Y-%m-%d").date() 

print(f"Run as date:    {run_as_date}")

In [0]:
config_path = "config/config.yml"
if not os.path.exists(config_path):
    raise FileNotFoundError(f"Missing configuration: {config_path}") 

with open(config_path, "r") as config_file:
    config = yaml.load(config_file, Loader=yaml.FullLoader)


RUN_NAME                            = config["shared"]["run_name"]          # "prod_2025_09_28"
#READ_ONLY_BUCKET                    = config["shared"]["read_only_bucket"]  # "memberanalytics-data-in-prod"
HIGH_FREQUENCY_VISITS_LAST_12_WEEKS = config["shared"]["high_frequency_visit_threshold"] # 12
LOW_FREQUENCY_VISITS_LAST_26_WEEKS  = config["shared"]["low_frequency_visit_threshold"]   # 0
START_WEEK_WINDOW                   = config["etl"]["start_week_windows"] 
BBM_WEEK_WINDOW                     = config["etl"]["bbm_week_window"] # 3
NUMBER_OF_WEEKS_TO_SAMPLE_FROM_EACH_CUSTOMER = config["etl"]["weeks_to_sample"] # 2
LAST_FISCAL_WEEK_TRAINING           = config["etl"]["end_date"]
FIRST_FISCAL_WEEK_TRAINING          = config["etl"]["start_date"]
BBM                                 = config["etl"]["bbm"] # false
OUTLIER_COLUMN                      = config["etl"]["outlier_column"] #"FW_SPEND_IN_STORE"
SEGMENTS_TO_FILTER_FOR              = [4, 5, 6, 7, 8, 10]


BUCKET                              = config["shared"]["bucket"]            # "memberanalytics-data-out-prod"
BBM_WEEKS_PATH  = "s3://{}/{}".format(BUCKET, config["etl"]["BBM_weeks_path"])
# "propensity_model_for_trips/input_files/mailer_cm_map.csv"

In [0]:
# Reads fiscal calendar data and "BBM weeks" (special business weeks) from S3.
# Joins and calculates a list of dates for 6 weeks prior to BBM weeks.
# Returns these as a list of week dates relevant for filtering and time windows in the ETL.

def get_bbm_weeks():
    lookup = spark.table(silver_fiscal_days)
    lookup = lookup.select("FISCAL_DAY", "FISCAL_WEEK_END")
    lookup = lookup.withColumn("date", f.to_date(lookup.FISCAL_DAY, "yyyy-MM-dd")    )
    lookup = lookup.withColumn("fiscal_week_end", f.to_date(lookup.FISCAL_WEEK_END, "yyyy-MM-dd")    )

    bbm_dates = spark.read.csv(path=BBM_WEEKS_PATH, inferSchema=True, header=True) 
    bbm_dates = bbm_dates.withColumn("date", f.to_date(bbm_dates.min_start_date, "MM/dd/yyy")    )

    combined = bbm_dates.join(lookup, ["date"], "left_outer")
    combined = combined.withColumn("6_weeks_prior_to_bbm", f.date_sub(lookup.fiscal_week_end, 42)    )
    weeks = [i["6_weeks_prior_to_bbm"] for i in combined.collect()]
    weeks = [week.strftime("%Y-%m-%d") for week in weeks]
    return weeks

In [0]:
BBM_CREATION_DATES = get_bbm_weeks()

# read in cube
print("----------------1/4 read in customer cube------------------")

customers = spark.table(fs_customer_cube_full) 

customers = util_func.create_independent_variables(
    customers,
    LOW_FREQUENCY_VISITS_LAST_26_WEEKS,
    HIGH_FREQUENCY_VISITS_LAST_12_WEEKS,
)

customers = util_func.create_seasonality_fields(customers)

customers = util_func.create_dependent_variables(
    customers, START_WEEK_WINDOW, BBM_WEEK_WINDOW, "bin"
)
customers = util_func.create_dependent_variables(
    customers, START_WEEK_WINDOW, BBM_WEEK_WINDOW, "cont"
)

# customer_cube_archive below is the archive version of the fs_merge delta table defined in config/variables
last_date = spark.table(customer_cube_archive).agg({"FISCAL_WEEK_END": "max"}).collect()[0][0]

customers = util_func.filter_customer_cube(
    customers,
    LAST_FISCAL_WEEK_TRAINING,
    FIRST_FISCAL_WEEK_TRAINING,
    BBM_CREATION_DATES,
    START_WEEK_WINDOW,
    BBM_WEEK_WINDOW,
    last_date,
)

customers = util_func.remove_outliers(customers, OUTLIER_COLUMN)

# Take a subset of oberservaitions
customers = util_func.subset_oberservations(
    customers, NUMBER_OF_WEEKS_TO_SAMPLE_FROM_EACH_CUSTOMER
)

# Match subset to segments to get the segments of the customer in the given week

In [0]:
customers.write.mode("overwrite").saveAsTable(model_trip_spend_etl_intermediate)

In [0]:
df_w_additional_columns = customers.select(
    "*",
    f.lit(run_as_date).cast("date").alias("run_date"),
    f.lit(RUN_NAME).cast("string").alias("run_name"),
    f.lit(LAST_FISCAL_WEEK_TRAINING).cast("string").alias("last_fiscal_week_training"),
    f.lit(FIRST_FISCAL_WEEK_TRAINING).cast("string").alias("first_fiscal_week_training")
)

df_w_additional_columns.write.mode("overwrite").option(
    "replaceWhere",
    f"run_date = '{run_as_date}' AND run_name = '{RUN_NAME}' AND last_fiscal_week_training = '{LAST_FISCAL_WEEK_TRAINING}' AND first_fiscal_week_training = '{FIRST_FISCAL_WEEK_TRAINING}'"
).saveAsTable(model_trip_spend_etl_intermediate_archive)
